# Inspect the trained team-stats model

Loads the Booster saved by `training/train.py` and answers two questions:

1. Which features contributed most to predictions?
2. What do the individual decision trees look like?

## 1. Load the saved Booster

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import xgboost as xgb

MODEL_PATH = Path('..') / 'serving' / 'models' / 'model.ubj'

model = xgb.Booster()
model.load_model(str(MODEL_PATH))
print(f'Loaded {MODEL_PATH}')
print(f'Features: {model.feature_names}')
print(f'Number of trees: {model.num_boosted_rounds()}')

## 2. Feature importance

Three importance types worth comparing:

- **gain** — average loss reduction when a feature is used in a split. Usually the most meaningful.
- **weight** — how often the feature appears in a split (count).
- **cover** — average number of samples touched by splits on that feature.

In [ ]:
import pandas as pd

rows = []
for itype in ('gain', 'weight', 'cover'):
    scores = model.get_score(importance_type=itype)
    for feat, score in scores.items():
        rows.append({'feature': feat, 'type': itype, 'score': score})

importance_df = (
    pd.DataFrame(rows)
    .pivot(index='feature', columns='type', values='score')
    .fillna(0)
    .sort_values('gain', ascending=False)
)
importance_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
xgb.plot_importance(model, importance_type='gain', ax=ax, title='Feature importance (gain)')
plt.tight_layout()
plt.show()

## 3. Visualize a decision tree

`to_graphviz` renders inline in the notebook (requires the `graphviz` system package — `brew install graphviz`).
Pick a tree by index; with `num_boost_round=50` you have trees `0..49`.

In [ ]:
TREE_INDEX = 0
xgb.to_graphviz(model, num_trees=TREE_INDEX)

### Text dump (no graphviz needed)

Each line is `node_id:[split condition] yes=… no=…` or a leaf with its score.

In [ ]:
dump = model.get_dump(with_stats=True)
print(f'Tree {TREE_INDEX}:\n')
print(dump[TREE_INDEX])